# Can a policy beat its teacher under stress?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccnets-team/causal-gpt-rl/blob/main/examples/can_a_policy_beat_its_teacher_under_stress.ipynb)

Two policies walk the same Humanoid.

The **parent** is Farama's SB3/TQC agent, five million steps of online
reinforcement learning in this environment. It is the policy that recorded the
Minari `mujoco/humanoid/medium-v0` trajectories.

The **child** is the `ccnets/causal-gpt-rl/humanoid-v5` bundle. It never
interacted with the environment. Everything it knows about walking, it read off
the trajectories the parent left behind.

On the environment they share, the parent is ahead — which is what you would
expect from a policy that practised in it against one that only read about it.

So this notebook changes the environment. Ground friction goes up by half, from
the first step, and neither policy is told. Nothing about that change appears in
the 348 numbers either of them observes; it is visible only in how the world
answers what they do.

The parent has no memory of the answers. The child has a KV cache.


## Install

`mujoco` is pinned to `3.2.3`. A different simulator release is a different
measurement even with identical weights and seeds, so the numbers below are
defined on this one.


In [ ]:
%pip install -q "causal-gpt-rl[hub,mujoco]" "mujoco==3.2.3"


## Get the repository

The rollout loop comes from
[`examples/deploy/reproduce.py`](https://github.com/ccnets-team/causal-gpt-rl/blob/main/examples/deploy/reproduce.py),
a file of this repository rather than part of the installed package. On Colab,
clone it; in a checkout, this cell does nothing.


In [ ]:
from pathlib import Path

if not Path("examples/deploy/reproduce.py").is_file():
    !git clone -q https://github.com/ccnets-team/causal-gpt-rl.git
    %cd causal-gpt-rl


## The parent

`farama-minari/Humanoid-v5-TQC-medium` publishes the actor as a plain
`policy.pth`, so this notebook reads the weights directly instead of installing
`stable-baselines3` and its own `gymnasium` pin beside the one the measurement is
defined on.

The actor is two hidden layers. SB3 squashes its output and then rescales it onto
the action space, and for Humanoid's `[-0.4, 0.4]` that whole tail reduces to
`0.4 * tanh(mu)`. The published repository carries no `vecnormalize.pkl`, so
observations go in raw.

The card reports **8021.95 ± 912.19**; the cell after next checks this
reimplementation against that number before anything is compared to it.


In [ ]:
import numpy as np
import torch
from huggingface_hub import hf_hub_download


class TQCRunner:
    """The published TQC actor, deterministic, over a batch of environments.

    `reset` / `act` / `observe` mirror `PolicyRunner`, so one rollout loop drives
    either policy. Being memoryless is the point of the comparison, so `observe`
    only keeps the latest observation and there is nothing to reset.
    """

    def __init__(self, repo_id, subdir, *, device="cpu", low=-0.4, high=0.4):
        path = hf_hub_download(
            repo_id=repo_id, filename=f"{subdir}/policy.pth",
            local_dir="tqc-parent",
        )
        sd = torch.load(path, map_location="cpu", weights_only=True)
        self.device = torch.device(device)
        self.low, self.high = float(low), float(high)

        def linear(prefix):
            w, b = sd[f"{prefix}.weight"], sd[f"{prefix}.bias"]
            layer = torch.nn.Linear(w.shape[1], w.shape[0])
            with torch.no_grad():
                layer.weight.copy_(w)
                layer.bias.copy_(b)
            return layer.to(self.device).eval()

        self.fc1 = linear("actor.latent_pi.0")
        self.fc2 = linear("actor.latent_pi.2")
        self.mu = linear("actor.mu")
        self._obs = None

    @torch.no_grad()
    def act(self, state=None):
        obs = self._obs if state is None else state
        x = torch.as_tensor(np.asarray(obs, dtype=np.float32), device=self.device)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        a = torch.tanh(self.mu(x))
        a = self.low + 0.5 * (a + 1.0) * (self.high - self.low)
        return a.cpu().numpy().astype(np.float32)

    def reset(self, initial_state):
        self._obs = np.asarray(initial_state)

    def observe(self, state):
        self._obs = np.asarray(state)


## The measurement

Fifty episodes, seeds 0..49, run as one fifty-row batch, capped at a thousand
steps — the protocol the published scores use. Every condition runs those same
seeds.

Friction is a property of the simulator's model rather than of its state, so
scaling it once after the environments are built holds for every step of every
episode: `reset` re-initialises positions and velocities, not the model.


In [ ]:
import gymnasium as gym

from causal_gpt_rl.inference import load_runner_from_hub
from examples.deploy.reproduce import (
    installed_versions,
    print_stack_report,
    run_seed_batch,
)

ENV_ID = "Humanoid-v5"
REPO_ID, SUBFOLDER = "ccnets/causal-gpt-rl", "humanoid-v5"
PARENT_REPO, PARENT_DIR = (
    "farama-minari/Humanoid-v5-TQC-medium", "humanoid-v5-TQC-medium",
)

CONTEXTS = [1, 32, 128, 1000]   # the bundle's own trained window is 32
FRICTIONS = [1.0, 1.5]          # 1.0 is the ground both policies know
EPISODES, MAX_STEPS = 50, 1000

seeds = list(range(EPISODES))
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"parent {PARENT_REPO}\nchild  {REPO_ID}/{SUBFOLDER}\non {ENV_ID}  ({device})\n")
print_stack_report(installed_versions())


def make_envs(friction_scale):
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make(ENV_ID) for _ in seeds],
        autoreset_mode=gym.vector.AutoresetMode.SAME_STEP,
    )
    for env in envs.envs:
        model = env.unwrapped.model
        # Column 0 of geom_friction is sliding friction; both defaults are 1.0.
        model.geom_friction[:, 0] = model.geom_friction[:, 0] * friction_scale
    return envs


def evaluate(policy, friction_scale):
    envs = make_envs(friction_scale)
    try:
        return run_seed_batch(envs, policy, seeds, MAX_STEPS, False)
    finally:
        envs.close()


## Check the parent before trusting it

A reimplemented forward pass that is subtly wrong would make the rest of this
notebook a comparison against a crippled baseline, which is the easiest way to
win an argument and the least interesting. This must land on the card's
8021.95 ± 912.19 — a different fifty seeds, so close rather than equal.


In [ ]:
parent = TQCRunner(PARENT_REPO, PARENT_DIR, device=device)
returns, lengths, ends = evaluate(parent, 1.0)
print(f"parent on clean ground: {returns.mean():.2f} +/- {returns.std():.2f}"
      f"   card: 8021.95 +/- 912.19")


## Run the comparison

Five policies — the parent, then the child at four retention settings — each on
both grounds. Ten batches of fifty episodes; expect roughly ten minutes on a
recent GPU, and considerably longer on CPU.

`kv_cache_max_len` is a load-time argument, so the four child runs load the same
weights and differ by one number.


In [ ]:
results = []
conditions = [("parent (TQC)", None)] + [("child", c) for c in CONTEXTS]

for label, context in conditions:
    for friction in FRICTIONS:
        policy = (
            TQCRunner(PARENT_REPO, PARENT_DIR, device=device)
            if context is None
            else load_runner_from_hub(
                repo_id=REPO_ID, subfolder=SUBFOLDER, device=device,
                num_envs=EPISODES, kv_cache_max_len=context,
            )
        )
        returns, lengths, ends = evaluate(policy, friction)
        # `terminated` is the Humanoid falling. `truncated` is the thousand-step
        # limit arriving with it still upright — that one is not a failure.
        fell = int(ends["terminated"].sum())
        results.append({
            "label": label, "context": context, "friction": friction,
            "returns": returns, "fell": fell, "upright": EPISODES - fell,
        })
        name = label if context is None else f"{label} ctx={context}"
        print(f"{name:<18} friction={friction:<4} "
              f"return={returns.mean():8.2f} +/- {returns.std():7.2f}  "
              f"fell={fell:>2}/{EPISODES}", flush=True)


## The table

Return is the headline, but on this environment the column that carries the
result is **fell** — how many of the fifty Humanoids hit the ground before the
thousand steps ran out.


In [ ]:
def row_for(label, context, friction):
    return next(r for r in results
                if r["label"] == label and r["context"] == context
                and r["friction"] == friction)


header = (f"{'policy':>16}   {'clean return':>13} {'fell':>6}   "
          f"{'friction 1.5':>13} {'fell':>6}   {'kept':>6}")
rule = "-" * len(header)
print(header)
print(rule)
for label, context in conditions:
    clean, stressed = row_for(label, context, 1.0), row_for(label, context, 1.5)
    kept = 100.0 * stressed["returns"].mean() / clean["returns"].mean()
    name = label if context is None else f"child ctx={context}"
    print(f"{name:>16}   {clean['returns'].mean():13.2f} "
          f"{str(clean['fell']) + '/' + str(EPISODES):>6}   "
          f"{stressed['returns'].mean():13.2f} "
          f"{str(stressed['fell']) + '/' + str(EPISODES):>6}   {kept:5.1f}%")
print(rule)


## What we measured

Fifty episodes per cell, seeds 0..49, MuJoCo 3.2.3 / Gymnasium 1.2.3 /
torch 2.8.0. Notebooks here are committed without outputs.

| policy | clean return | fell | friction 1.5 | fell | kept |
|:---|---:|---:|---:|---:|---:|
| parent — TQC, 5M steps | **8103.30** | **2 / 50** | 4359.99 | 42 / 50 | 53.8% |
| child `ctx=1` | 7228.38 | 9 / 50 | 3778.90 | 39 / 50 | 52.3% |
| child `ctx=32` — *trained window* | 7572.63 | 7 / 50 | **5693.00** | **27 / 50** | **75.2%** |
| child `ctx=128` | 8037.26 | 2 / 50 | 5322.56 | 37 / 50 | 66.2% |
| child `ctx=1000` | 7389.61 | 9 / 50 | 5022.97 | 37 / 50 | 68.0% |

The clean column reproduces the published table in
[`can_longer_context_help_humanoid.ipynb`](https://github.com/ccnets-team/causal-gpt-rl/blob/main/examples/can_longer_context_help_humanoid.ipynb)
to the last decimal — same protocol, same seeds, a different notebook.

### On the ground both policies know

The parent is ahead, and it should be. It practised here for five million steps.
A paired bootstrap over the fifty shared seeds puts `ctx=32` **531 return behind
it** (95% CI [-994, -144]), which is a real gap rather than a draw.

The one child setting that catches it is `ctx=128`: **-66** against the parent,
95% CI [-174, +103]. That interval straddles zero, so the honest reading is a
tie, not a win. Both put two Humanoids on the ground out of fifty.

A policy that only ever read the parent's logs draws level with it. It does not
pass it.

### On ground neither of them practised on

Raising sliding friction by half costs the parent **46% of its return** and puts
**42 of 50** Humanoids on the ground.

The child at its trained window keeps **75%** and falls **27** times. Against the
parent that is **+1333 return**, 95% CI [+145, +2481] — significant, and the
direction reverses from the clean column.

The mechanism shows up in the retention axis. `ctx=1` keeps one step of past,
which is as close to memoryless as this runner gets, and it lands with the
parent: **-581** against it, CI [-1911, +755], indistinguishable. Against
`ctx=32` it is **-1914**, CI [-3130, -669] — significant. What separates the
child from the parent under stress is the retained past, not the weights.

### Where the peak is, we do not know

`ctx=128` minus `ctx=32` under stress is **-370**, CI [-1462, +740]. Fifty seeds
cannot order them. Repeating this at a hundred episodes moved the best stressed
setting from 32 to 128 while every conclusion above held, so read the table as
*retention helps under stress* and not as *32 is the number*.

Nothing about the friction change reaches either policy's observation vector. The
child does not detect it and compensate; it carries enough recent history that
what the ground does differently is already in the window it is conditioning on.


## What this shows, and what it does not

**One environment, one perturbation, one direction.** Friction going *up* is what
breaks a Humanoid; going down to 0.5 barely troubles either policy. Mass is
harsher still — a Humanoid at 0.8x its own mass falls in every episode we ran, so
there is no partial-failure band there to measure in. This result is about one
axis, and it is a measurement to repeat rather than a number to carry over.

**The parent is not being criticised.** It practised in this environment for five
million steps and it walks better there than the policy that only read its logs.
That ordering is the honest starting point, and it is why the interesting
question is what happens when the environment stops being the one it practised in.

**Retention is not free and not monotone.** `ctx=1` is, for this purpose,
memoryless, and it lands with the parent. Past the peak the advantage narrows
again. More retained past is not automatically better — there is a setting, and
it has to be found.

**Fifty seeds is one draw.** Differences of a few hundred return do not survive a
reshuffle; changes in how many Humanoids stayed upright do. Treat the `fell`
column as the result and the means as the context around it.

**Nothing here was retrained.** The four child runs load one file and change one
load-time argument.

To measure a bundle under the published protocol on the ground it knows:

```
python -m examples.deploy.reproduce --env-id Humanoid-v5 --kv-cache-max-len 128
```

To sweep sensor and actuator noise instead of the ground:

```
python -m examples.deploy.noise --env-id Humanoid-v5 --channel obs --dims 0:45
```
